In [ ]:
from proto import *
from engine import *

import pandas as pd
import numpy as np

pd.options.plotting.backend = "plotly"

In [ ]:
pop_strat = Stratification("pop", ["human", "mosquito"])
pops = CompartmentMap.new(pop_strat)
pops.compartments

In [ ]:
h_disease_state = Stratification("disease_state", ["S", "E", "I", "R"])
pops.stratify(h_disease_state, pop_strat["human"])
pops

In [ ]:
m_disease_state = Stratification("disease_state", ["S", "E", "I"])
pops.stratify(m_disease_state, pop_strat["mosquito"])
pops

In [ ]:
def foi_m2h(cdatamap, params):
    infected_m = cdatamap.query(m_disease_state["I"]).data.sum()
    return infected_m * params["bite_rate"]


def foi_h2m(cdatamap, params):
    infected_h = cdatamap.query([h_disease_state["I"]]).data.sum()
    total_h = cdatamap.query([pop_strat["human"]]).data.sum()
    return (infected_h / total_h) * params["bite_rate"]

In [ ]:
infection_h = TransitionFlow(h_disease_state["S"], h_disease_state["E"], "foi_m2h")
progression_h = TransitionFlow(
    h_disease_state["E"], h_disease_state["I"], "h_progression_rate"
)
recovery_h = TransitionFlow(
    h_disease_state["I"], h_disease_state["R"], "h_recovery_rate"
)
waning_h = TransitionFlow(h_disease_state["R"], h_disease_state["S"], "h_waning_rate")

In [ ]:
infection_m = TransitionFlow(m_disease_state["S"], m_disease_state["E"], "foi_h2m")
progression_m = TransitionFlow(
    m_disease_state["E"], m_disease_state["I"], "m_progression_rate"
)

death_m = ExitFlow(pop_strat["mosquito"], "m_death_rate")

In [ ]:
def m_birth_seasonal(cdatamap, params):
    t = params["t"]
    return 0.5 * (1.0 + jnp.sin(np.pi * 2.0 * t / 365.25)) * params["m_birth_scale"]


birth_m = EntryFlow(m_disease_state["S"], "m_birth")

In [ ]:
flows = {
    "infection_h": infection_h,
    "progression_h": progression_h,
    "recovery_h": recovery_h,
    "infection_m": infection_m,
    "progression_m": progression_m,
    "death_m": death_m,
    "birth_m": birth_m,
    "waning_h": waning_h,
}

dyn_params = {"foi_m2h": foi_m2h, "foi_h2m": foi_h2m, "m_birth": m_birth_seasonal}

model = NaiveModel(pops, flows, dyn_params)
run = model.get_runner()

In [ ]:
pops.compartments

In [ ]:
istate = jnp.array([100.0, 0.0, 0.0, 0.0, 100.0, 0.0, 1.0])
params = {
    "h_progression_rate": 0.1,
    "h_recovery_rate": 0.05,
    "h_waning_rate": 0.01,
    "m_progression_rate": 0.5,
    "m_death_rate": 0.03,
    "m_birth_scale": 1.0,
    "bite_rate": 0.015,
}

t = 1000

comp_results = run(istate, params, t)

In [ ]:
def compname(c: Compartment):
    return "_".join([stratum for (strat, stratum) in c.strata])


comp_labels = [compname(c) for c in model.cmap.compartments]

pd.DataFrame(comp_results, columns=comp_labels).plot()